### Google Sheets read (service account)

1. Enable **Google Sheets API** and **Google Drive API** on your GCP project.
2. Create a service account, download a JSON key, and **share the spreadsheet** with the service account email (`…@….iam.gserviceaccount.com`) as Viewer (or Editor).
3. Set `GOOGLE_APPLICATION_CREDENTIALS` to the key file path, or set `credentials_path` explicitly in the next cell.

Install deps: `uv sync --extra google-sheets`

In [1]:
import os
from pathlib import Path

import gspread
import pandas as pd
from google.oauth2.service_account import Credentials

# From URL: https://docs.google.com/spreadsheets/d/<SPREADSHEET_ID>/edit
SPREADSHEET_ID = "1DynAAKo8sHkGx6TrfSQDlaqyQ5N8jcThXFWGh7k1IlQ"
SHEET_NAME = "1.Clientlist"

# credentials_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
credentials_path = "./config/gsheet-creds.json"
# Or set explicitly: credentials_path = Path.home() / "secrets" / "service-account.json"
if not credentials_path:
    raise FileNotFoundError(
        "Set GOOGLE_APPLICATION_CREDENTIALS to your service account JSON path"
    )
credentials_path = Path(credentials_path).expanduser()

In [2]:
scopes = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]
creds = Credentials.from_service_account_file(str(credentials_path), scopes=scopes)
gc = gspread.authorize(creds)
sh = gc.open_by_key(SPREADSHEET_ID)

In [3]:
ws = sh.worksheet(SHEET_NAME)
records = ws.get_all_records()
df = pd.DataFrame(records)

# If the first row is not a clean header (duplicates / blanks), use:
# values = ws.get_all_values()
# df = pd.DataFrame(values[1:], columns=values[0])

df

,easybill_kundennummer,last_invoice_date,€ net billed,spe. care,wochenliste_ids,easybill_firma,easybill_name,easybill_vorname,easybill_address,medisoft_ids,medisoft_names,sim_scores,zoho_id,link to zoho,validated,no_migration
0,130002124,2025-09-09,"311,00 €",TRUE,,.change GmbH,,,Tersteegenstr. 25 40474 Düsseldorf,00_A1W00VHI68\n00_8JJ00VTGCE,BSH Düsseldorf / .change GmbH\nChange,1.0\n1.0,,FALSE,FALSE,FALSE
1,130000200,2026-01-02,"27 547,00 €",TRUE,130000200,"""Auf der Tenne"" e.V.",,,Pankelower Weg 13a 18196 Dummerstorf,,,,zcrm_386758000010045564,👉 link to zoho account,FALSE,FALSE
2,130000437,2023-01-19,"1 622,00 €",FALSE,,"""Das Futterhaus"" - Berlin GmbH & Co. KG",,,Lankwitzer Str. 3 12107 Berlin,,,,zcrm_386758000025371007,👉 link to zoho account,FALSE,FALSE
3,130002066,2025-08-29,"143,00 €",TRUE,,„Die 3“ Transport- und Handelsgesellschaft mbH,,,An der Mühle 2 23972 Dorf Mecklenburg,00_A0S00RIHE7,BSH Rostock / Die 3 - Transport- und Handelsge...,1.0,,FALSE,FALSE,FALSE
4,111000009,2021-04-15,"2 604,00 €",TRUE,,"""K"" Line (Deutschland) GmbH",,,Anckelmannsplatz 1 20537 Hamburg,00_8PG00L8MCI,BSH Hamburg / K Line Deutschland GmbH,1.0,zcrm_386758000010019168,👉 link to zoho account,FALSE,FALSE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3207,130001596,2026-01-02,"2 532,00 €",TRUE,130001596,,Ragg,Johann C.,Bayreuther Str. 36 10789 Berlin,00_9Z700SML7M,,,zcrm_386758000048508055,👉 link to zoho account,FALSE,FALSE
3208,130001595,2026-01-05,"1 775,00 €",FALSE,130001595,,Sonne,Jan-Ulrich,Oststraße 51 40211 Düsseldorf,,,,zcrm_386758000049827165,👉 link to zoho account,FALSE,FALSE
3209,130002179,2026-01-09,"2 075,00 €",FALSE,130002179,,Bacher,Siegfried,Kolonnenstraße 56 10827 Berlin,,,,zcrm_386758000057758947,👉 link to zoho account,FALSE,FALSE
3210,130000457,2026-01-12,"1 063,00 €",TRUE,,,Rindler,Norman,Schlossgartenallee 21 19061 Schwerin,,,,,FALSE,FALSE,FALSE


In [ ]:
# Optional: export to CSV (paths relative to notebook kernel cwd — often repo root)
output_csv = Path("firms_join_table/spreadsheet_extract/extracted.csv")
output_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_csv, index=False)
output_csv

### Same sheet via DuckDB (`gsheets` extension)

Uses the **same service-account JSON** as gspread, but with DuckDB’s [`read_gsheet`](https://duckdb.org/community_extensions/extensions/gsheets.html) so you can query the tab in SQL (`CREATE TABLE … AS`, joins, etc.).

1. Run the **first code cell** above so `credentials_path`, `SPREADSHEET_ID`, and `SHEET_NAME` are defined (or redefine them here).
2. First run needs network: `INSTALL gsheets FROM community`.
3. Auth: `CREATE SECRET … PROVIDER key_file, FILEPATH '…'` — see [duckdb-gsheets.com](https://duckdb-gsheets.com/).

Alternative: interactive OAuth — `CREATE SECRET (TYPE gsheet);` (browser + token paste), no JSON file.

In [3]:
import duckdb

# Run the first gspread config cell so `credentials_path`, `SPREADSHEET_ID`, and `SHEET_NAME` exist.


def _sql_literal(s: str) -> str:
    return s.replace("'", "''")


key_path = _sql_literal(str(credentials_path.resolve()))

con = duckdb.connect()
con.execute("INSTALL gsheets FROM community;")
con.execute("LOAD gsheets;")
con.execute(
    f"""
CREATE OR REPLACE SECRET gsheet_sa (
    TYPE gsheet,
    PROVIDER key_file,
    FILEPATH '{key_path}'
);
"""
)

In [5]:
# all_varchar=true avoids cast errors when Sheets mixes text/emoji/booleans in one column
df_duck = con.sql(
    f"""
SELECT * FROM read_gsheet(
    '{_sql_literal(SPREADSHEET_ID)}',
    sheet='{_sql_literal(SHEET_NAME)}',
    all_varchar=true
)
"""
).df()
df_duck

,easybill_kundennummer,last_invoice_date,€ net billed,spe. care,wochenliste_ids,easybill_firma,easybill_name,easybill_vorname,easybill_address,medisoft_ids,medisoft_names,sim_scores,zoho_id,link to zoho,validated,no_migration
0,130002124,2025-09-09,"311,00 €",TRUE,NaN,.change GmbH,NaN,NaN,Tersteegenstr. 25 40474 Düsseldorf,00_A1W00VHI68\n00_8JJ00VTGCE,BSH Düsseldorf / .change GmbH\nChange,1.0\n1.0,NaN,FALSE,FALSE,FALSE
1,130000200,2026-01-02,"27 547,00 €",TRUE,130000200,"""Auf der Tenne"" e.V.",NaN,NaN,Pankelower Weg 13a 18196 Dummerstorf,NaN,NaN,NaN,zcrm_386758000010045564,👉 link to zoho account,FALSE,FALSE
2,130000437,2023-01-19,"1 622,00 €",FALSE,NaN,"""Das Futterhaus"" - Berlin GmbH & Co. KG",NaN,NaN,Lankwitzer Str. 3 12107 Berlin,NaN,NaN,NaN,zcrm_386758000025371007,👉 link to zoho account,FALSE,FALSE
3,130002066,2025-08-29,"143,00 €",TRUE,NaN,„Die 3“ Transport- und Handelsgesellschaft mbH,NaN,NaN,An der Mühle 2 23972 Dorf Mecklenburg,00_A0S00RIHE7,BSH Rostock / Die 3 - Transport- und Handelsge...,1.0,NaN,FALSE,FALSE,FALSE
4,111000009,2021-04-15,"2 604,00 €",TRUE,NaN,"""K"" Line (Deutschland) GmbH",NaN,NaN,Anckelmannsplatz 1 20537 Hamburg,00_8PG00L8MCI,BSH Hamburg / K Line Deutschland GmbH,1.0,zcrm_386758000010019168,👉 link to zoho account,FALSE,FALSE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3207,130001596,2026-01-02,"2 532,00 €",TRUE,130001596,NaN,Ragg,Johann C.,Bayreuther Str. 36 10789 Berlin,00_9Z700SML7M,NaN,NaN,zcrm_386758000048508055,👉 link to zoho account,FALSE,FALSE
3208,130001595,2026-01-05,"1 775,00 €",FALSE,130001595,NaN,Sonne,Jan-Ulrich,Oststraße 51 40211 Düsseldorf,NaN,NaN,NaN,zcrm_386758000049827165,👉 link to zoho account,FALSE,FALSE
3209,130002179,2026-01-09,"2 075,00 €",FALSE,130002179,NaN,Bacher,Siegfried,Kolonnenstraße 56 10827 Berlin,NaN,NaN,NaN,zcrm_386758000057758947,👉 link to zoho account,FALSE,FALSE
3210,130000457,2026-01-12,"1 063,00 €",TRUE,NaN,NaN,Rindler,Norman,Schlossgartenallee 21 19061 Schwerin,NaN,NaN,NaN,NaN,FALSE,FALSE,FALSE


In [ ]:
# Optional: load once into a DuckDB table for faster repeated queries
con.execute(
    f"""
CREATE OR REPLACE TABLE sheet_clientlist AS
SELECT * FROM read_gsheet(
    '{_sql_literal(SPREADSHEET_ID)}',
    sheet='{_sql_literal(SHEET_NAME)}',
    all_varchar=true
);
"""
)
con.sql("SELECT count(*) AS n FROM sheet_clientlist").df()

In [19]:
from merge_tables.db.connection import connect_to_postgres_via_duckdb

In [20]:
duck = connect_to_postgres_via_duckdb()

✓ Successfully connected DuckDB to PostgreSQL database 'medisoft'


In [7]:
duck.sql(
    """
    select csv.*, c."Kontakt: Firma" as easybill_firm, f.pfad as medisoft_firm
    from read_csv('output/easybill_medisoft_pairs.csv') csv
    left join pg.easybill.contacts c on replace(easybill_kundennummer::varchar, ' ', '') = c."Kontakt: Kundennummer"
    left join pg.medisoft.table_firmenstruktur f on medisoft_id = f.rec_id
    """
).to_csv("output/easybill_medisoft_pairs_with_firm_path.csv")

In [8]:
duck.execute("INSTALL gsheets FROM community;")
duck.execute("LOAD gsheets;")
duck.execute(
    f"""
CREATE OR REPLACE SECRET gsheet_sa (
    TYPE gsheet,
    PROVIDER key_file,
    FILEPATH '{key_path}'
);
"""
)

In [47]:
duck.sql(
    """
    select 
        easybill_kundennummer,
        last_invoice_date,
        "€ net billed",
        "spe. care",
        wochenliste_ids,
        ur."BAS Standort",
        easybill_firma,
        easybill_address,
        medisoft_id,
        m.pfad as medisoft_firm,
        m.strasse as medisoft_street, m.ort
    from read_gsheet(
        '1DynAAKo8sHkGx6TrfSQDlaqyQ5N8jcThXFWGh7k1IlQ',
        sheet='1.Clientlist',
        all_varchar=true
    )
    left join read_csv('output/easybill_medisoft_pairs.csv') csv
        using(easybill_kundennummer)
    left join pg.medisoft.table_firmenstruktur m
        on medisoft_id = m.rec_id
    left join read_gsheet(
        '1CB4_x7nnuxuqkakx4so0UJU9ZIRwy_htOzZ3Kc2110g',
        sheet='ÜR Kunden',
        range='A2:Z',
        all_varchar=true
    ) ur on trim(ur."Kunde") = easybill_firma
    where '\n' in wochenliste_ids 
    order by csv.easybill_kundennummer
    """
)

┌───────────────────────┬───────────────────┬──────────────┬───────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────┬───────────────────────────────────────────────────────┬─────────────────────────────────────────────┬───────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────┬─────────┐
│ easybill_kundennummer │ last_invoice_date │ € net billed │ spe. care │                                                                                                                                         wochenliste_ids                                                                                                                                          │ B

In [ ]:
duck.sql(
    """
    begin;
    create or replace table pg.bas_firms.cleaned_medisoft as
    select 
        trim(medisoft_id) as medisoft_id,
        nullif(trim(name), '') as name,
        nullif(trim(kuerzel), '') as kuerzel,
        nullif(trim(pfad), '') as pfad,
        nb_patients::int as nb_patients,
        trim(last_exam_date) as last_exam_date,
        trim(addresse) as address,
        trim("has easybill connection") == 'TRUE' as has_easybill_connection,
        trim("migrate as inactive") == 'TRUE' as migrate_as_inactive,
        trim("no migration") == 'TRUE' as no_migration,
        trim("Selbstzahler") == 'TRUE' as selbstzahler,
        trim(city) as city
    from read_csv('output/medisoft_union.csv', all_varchar=true)
    where regexp_matches(medisoft_id, '(\\d+)') is true;
    commit;
    """
)

In [38]:
duck.sql(
    """
    select 
        regexp_matches(medisoft_id, '(\\d+)') as regex,
        city
    from read_csv('output/medisoft_union.csv')
    where regex is false
    """
)

┌─────────┬─────────────┐
│  regex  │    city     │
│ boolean │   varchar   │
├─────────┼─────────────┤
│ false   │  Viersen    │
│ false   │  Viersen    │
│ false   │  Viersen    │
│ false   │  Viersen    │
│ false   │  Viersen    │
│ false   │  Viersen    │
│ false   │  Viersen    │
│ false   │  Viersen    │
│ false   │  Viersen    │
│ false   │  Viersen    │
│   ·     │      ·      │
│   ·     │      ·      │
│   ·     │      ·      │
│ false   │  Viersen    │
│ false   │  Viersen    │
│ false   │  Viersen    │
│ false   │  Viersen    │
│ false   │  Viersen    │
│ false   │  Viersen    │
│ false   │  Viersen    │
│ false   │  Viersen    │
│ false   │  Viersen    │
│ false   │  Viersen    │
├─────────┴─────────────┤
│ 967 rows    2 columns │
│ (20 shown)            │
└───────────────────────┘

In [21]:
import duckdb

# Run the first gspread config cell so `credentials_path`, `SPREADSHEET_ID`, and `SHEET_NAME` exist.


def _sql_literal(s: str) -> str:
    return s.replace("'", "''")


key_path = _sql_literal(str(credentials_path.resolve()))

duck.execute("INSTALL gsheets FROM community;")
duck.execute("LOAD gsheets;")
duck.execute(
    f"""
CREATE OR REPLACE SECRET gsheet_sa (
    TYPE gsheet,
    PROVIDER key_file,
    FILEPATH '{key_path}'
);
"""
)

In [22]:
duck.sql(
    """
    create or replace table easybill_zoho as 
    select * from read_gsheet(
    '1oGLLTtudUdfeN0iF8HDgRsqPvjoaw42CvvIa8EDgQ1w', sheet='Feuille 1', all_varchar=true)
    """
)

In [23]:
duck.sql(
    """
    select 
    easybill_kundennummer,
    nullif(regexp_extract(zoho_url, 'Accounts/(\\d{18})', 1), '') as zoho_id,
    from easybill_zoho
    """
)

┌───────────────────────┬────────────────────┐
│ easybill_kundennummer │      zoho_id       │
│        varchar        │      varchar       │
├───────────────────────┼────────────────────┤
│ 130002124             │ NULL               │
│ 130000200             │ 386758000010045564 │
│ 130000437             │ 386758000025371007 │
│ 130002066             │ 386758000010023118 │
│ 111000009             │ 386758000010019168 │
│ 113010031             │ 386758000010041451 │
│ 130000392             │ 386758000025186099 │
│ 127000001             │ 386758000012489188 │
│ 127000000             │ 386758000010043421 │
│ 130000996             │ 386758000010018006 │
│     ·                 │  ·                 │
│     ·                 │  ·                 │
│     ·                 │  ·                 │
│ 130001038             │ NULL               │
│ 130001933             │ NULL               │
│ 119020052             │ NULL               │
│ 130000941             │ NULL               │
│ 116010039  

In [29]:
duck.execute(
    """
    rollback;
    begin;
    truncate table pg.bas_firms.easybill_zoho;
    insert into pg.bas_firms.easybill_zoho (easybill_id, zoho_id)
    select 
    easybill_kundennummer,
    nullif(regexp_extract(zoho_url, 'Accounts/(\\d{18})', 1), '') as zoho_id,
     from easybill_zoho;
    commit;
    """
)